# 0 导入函数

In [1]:
import os
# dotenv包是从.env 文件加载环境变量（如 API Key），避免将密钥硬编码在代码中
from dotenv import load_dotenv
# LangChain 社区提供的 PDF 加载器，它会读取 PDF 文件并解析成文本
# 返回 Document 对象列表（每个对象包含 page_content 和元数据）。
from langchain_community.document_loaders import PyPDFLoader
# 文本切分器，用于将长文档切分成适合向量化的小块（chunk）
# 它会递归地按分隔符（如换行、句号）切分，尽量保持语义完整
from langchain_text_splitters import RecursiveCharacterTextSplitter
# LangChain 对 OpenAI 风格 Embedding API 的封装，虽然后面直接去hugging face下载
from langchain_community.embeddings import OpenAIEmbeddings
# FAISS封装
from langchain_community.vectorstores import FAISS
# 调用ds接口
from openai import OpenAI
# 设置环境变量，把hugging face下载地址改成国内镜像，虽然后面没用
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

/tmp/ipykernel_1383085/1905265516.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


# 1 加载环境变量

In [6]:
# 1. 加载环境变量（确保 .env 里有 DEEPSEEK_API_KEY）
load_dotenv()
api_key = os.environ.get("DEEPSEEK_API_KEY")
if not api_key:
    raise ValueError("请在 .env 中设置 DEEPSEEK_API_KEY")

In [ ]:
# 2. 读取 PDF（请把pdf放在同目录下）
loader = PyPDFLoader("政治理论+精讲精练6.pdf") # 形成PDF分页列表
docs = loader.load()

Ignoring wrong pointing object 101 0 (offset 0)
Ignoring wrong pointing object 102 0 (offset 0)
Ignoring wrong pointing object 145 0 (offset 0)
Ignoring wrong pointing object 146 0 (offset 0)
Ignoring wrong pointing object 149 0 (offset 0)
Ignoring wrong pointing object 150 0 (offset 0)
Ignoring wrong pointing object 153 0 (offset 0)
Ignoring wrong pointing object 154 0 (offset 0)
Ignoring wrong pointing object 471 0 (offset 0)
Ignoring wrong pointing object 472 0 (offset 0)


In [10]:
# 3. 切分文档（每段 500 字符，重叠 50）
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, # 最大字符数
    chunk_overlap=50, # 片段重叠数，方便上下文对接
    separators=["\n\n", "\n", "。", "！", "？", "；", "，", " ", ""]
)
chunks = splitter.split_documents(docs)
print(f"✓ 成功切分为 {len(chunks)} 个片段")

✓ 成功切分为 14 个片段


# 4. Embedding  
embeddings：传入了本地文件夹的绝对路径。  
此时，程序只是“实例化”了一个转换器对象（翻译：只是告诉了它“你的模型文件在那个文件夹里”），但还没有加载到内存

In [13]:
# 4. 使用了hugging face本地embedding
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="bge-small-zh-v1.5")

/home/s502025300013/miniconda3/envs/ILD/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 71/71 [00:00<00:00, 18585.51it/s]


# 5. 构建 FAISS 向量库（存入本地，以便下次复用）
分为首次使用和构建完数据库后使用  
vectorstore =包括三个作用：  
- 加载模型：程序去指定的embedding文件夹把模型（model.safetensors 和分词器）全部载入内存。  
- 批量向量化：遍历 chunks 列表（比如有 50 个片段），对每个片段调用模型，把中文文字变成一串浮点数数组（比如 384 个数字）。这一步通常最吃 CPU/内存。  
- 构建索引：把所有生成的向量丢给 FAISS 库，FAISS 会把这些向量排列成一种特殊的“树形结构”或“图结构”（索引），方便 1 秒钟之内找到最相似的向量。  

vectorstore.save_local("faiss_index")：  
内存数据序列化（落盘）保存到硬盘。下次运行代码时，可以直接 FAISS.load_local() 读取，跳过前面漫长的向量化计算，直接进入检索阶段

In [16]:
# 5. 构建 FAISS 向量库（首次使用需要跑这步，存入本地，以便下次复用）
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("faiss_index")
print("✓ 向量库已构建并保存")

✓ 向量库已构建并保存


In [17]:
# 之后使用直接从硬盘读取索引
vectorstore = FAISS.load_local(
    "faiss_index", 
    embeddings, 
    allow_dangerous_deserialization=True
)

# 6. 检索 + 生成回答
1 .retriever = vectorstore.as_retriever(search_kwargs={"k": top_k})：去前面的vectorstore里包装成一个retriever工具包，能够找topk（在函数入口处传入）  
2. vectorstore是一个对象，是faiss的实例，等于数据库+工具包  
3. .invoke(question)：启动口令。执行时候吧question扔给embedding变成向量、去用knn算相似度、挑出top3文本打包成retrieved_docs变量  
4. [doc.page_content for doc in retrieved_docs]：把 retrieved_docs 列表里的每一个文档对象（doc）的文本内容（page_content）提取出来，组成一个新的纯文本列表。比如原来存的是 [文档A对象, 文档B对象]，变成了 ["段落A的文字", "段落B的文字"]  
5. 两个换行符 \n\n 作为胶水，把列表里的几个段落粘成一段长长的纯文本。最后context是字符串。  
6. system：简单调用可以不屑，但工程中绝对必须，能让模型不乱说

In [19]:
# 6. 检索 + 生成回答
def ask_question(question, top_k=3):

    # 检索最相关的 top_k 个片段
    retriever = vectorstore.as_retriever(search_kwargs={"k": top_k})
    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    # 拼装 prompt
    prompt = f"""基于以下文档内容回答问题。如果文档中没有相关信息，请直接说“未找到相关信息”。
文档内容：
{context}

问题：{question}
答案："""

    # 调用 DeepSeek Chat API
    client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com")
    response = client.chat.completions.create(
        model="deepseek-flash",
        messages=[
            {"role": "system", "content": "你是一个严谨的助理，只能基于提供的文档内容回答。"},
            {"role": "user", "content": prompt}
        ],
        stream=False
    )
    return response.choices[0].message.content

# 7 测试

In [20]:
# 7. 测试
if __name__ == "__main__":
    question = "什么是上层建筑？自然科学属于上层建筑吗？"
    answer = ask_question(question)
    print(f"\n问：{question}")
    print(f"答：{answer}")


问：什么是上层建筑？自然科学属于上层建筑吗？
答：根据文档，上层建筑是指建立在一定经济基础之上的意识形态以及与之相适应的制度、组织和设施。上层建筑系统可分为政治上层建筑和观念上层建筑。政治上层建筑包括政治法律制度，以及国家政权机构、政党、军队、警察、法庭、监狱等政治组织形态和设施；观念上层建筑即社会意识形态，包括政治法律思想、道德、宗教、哲学、艺术等思想观点。

关于“自然科学属于上层建筑吗”，文档中未提及自然科学是否属于上层建筑，未找到相关信息。
